<img src="https://whatcar.vn/media/2018/09/car-lot-940x470.jpg"/>

## ??????????????? ????????? ?????????? ?? ???????????????
*???? ??????? ???????? ????????/???????? (Baseline) ? ????? ???????????? ? ?? ?????? ??????? ????????!*   
?? ?????? ???????????? ??? ??? ?????? ??? ?????????? ?????? ???????.


> **baseline** ????????? ?????? ??? ??????, ??? ????? ?????????? ??? ?????????? ????????? ? ????????? ??????? ? ??? ????? ???????? ?? ??????. ??? ???? ?? ??????? ????? ???? ?????????? ???????. ??? ???????? ??????? ?????????? ? ?????? ??, ? ?? ??????? ?????? ????? ?? ????? ?????????? ??????. 
????? baseline ???????? ??????? ??????? ?????? ?? ???????. ???? ???? ??????? ???? baseline - ?? ???? ??????? ???-?? ?? ?? ? ????? ??????????? ?????? ????) 

??????, ??? ?? ??????? ????????????, ??? ????? ?????????????? ??????? ????????? ???????. ? ???? ???????? ?? ?? ????? ????????????? ???? ??????. ???????????, ??? ?? ??? ??? ??????? ? ?????? ?????????? ???? ??????? ????? "Add Data", ????? ?????????? ? ?????? ML.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sys
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder

In [ ]:
print('Python       :', sys.version.split('\n')[0])
print('Numpy        :', np.__version__)

In [ ]:
# ??????????? ?????? ???????, ????? ???????????? ???? ??????????????:
!pip freeze > requirements.txt

In [ ]:
# ?????? ?????????? RANDOM_SEED, ????? ???? ???????????? ???? ??????????????!
RANDOM_SEED = 42

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_pred-y_true)/y_true))

# Setup

In [ ]:
VERSION    = 16
DIR_TRAIN  = '../input/parsing-all-moscow-auto-ru-09-09-2020/' # ????????? ? ???????? ??????? ???????
DIR_TEST   = '../input/sf-dst-car-price-prediction/'
VAL_SIZE   = 0.20   # 20%

# Data

In [ ]:
!ls '../input'

In [ ]:
train = pd.read_csv(DIR_TRAIN+'all_auto_ru_09_09_2020.csv') # ??????? ??? ???????? ??????
test = pd.read_csv(DIR_TEST+'test.csv')
sample_submission = pd.read_csv(DIR_TEST+'sample_submission.csv')

In [ ]:
train.head(5)

In [ ]:
train.info()

In [ ]:
test.head(5)

In [ ]:
test.info()

## Data Preprocessing

In [ ]:
# ... 

In [ ]:
train.dropna(subset=['productionDate','mileage'], inplace=True)
train.dropna(subset=['price'], inplace=True)

In [ ]:
# ??? baseline ?????? ?????? ???? ?????? ????????? ??? ?????? ?????????
columns = ['bodyType', 'brand', 'productionDate', 'engineDisplacement', 'mileage']
df_train = train[columns]
df_test = test[columns]

In [ ]:
y = train['price']

## Label Encoding

In [ ]:
# ?????! ??? ?????????? ????????? ????????? ?????????? ????? ? ???? ? ???? ???????
df_train['sample'] = 1 # ???????? ??? ? ??? ?????
df_test['sample'] = 0 # ???????? ??? ? ??? ????

data = df_test.append(df_train, sort=False).reset_index(drop=True) # ??????????

In [ ]:
for colum in ['bodyType', 'brand', 'engineDisplacement']:
    data[colum] = data[colum].astype('category').cat.codes

In [ ]:
data

In [ ]:
X = data.query('sample == 1').drop(['sample'], axis=1)
X_sub = data.query('sample == 0').drop(['sample'], axis=1)

## Train Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=VAL_SIZE, shuffle=True, random_state=RANDOM_SEED)

# Model 1: ???????? "???????" ?????? 
??? ?????? ????? ????????????? ??????? ???? ?? ?????? ????????? (engineDisplacement). 
C ??? ????? ?????????? ?????? ??????.




In [ ]:
tmp_train = X_train.copy()
tmp_train['price'] = y_train

In [ ]:
# ??????? median ?? ??????????? engineDisplacement ? ?????? ? ????????? ????
predict = X_test['engineDisplacement'].map(tmp_train.groupby('engineDisplacement')['price'].median())

#????????? ????????
print(f"???????? ??????? ?????? ?? ??????? MAPE: {(mape(y_test, predict.values))*100:0.2f}%")

# # Model 2 : CatBoost
![](https://pbs.twimg.com/media/DP-jUCyXcAArRTo.png:large)   


? ??? ? ?????? ??????????? ??? ???????? ??????????????. ?????????? ??? ?????? ? ?????? ??????? ???? ??????? ????? ??????? ?????????? CatBoost ?? ???????. [https://catboost.ai](http://)     
?? ?????? ?????? **CatBoost ???????? ????? ?? ?????? ????????? ??? ????????? ??????!**

#### ???????? ????? ? CatBoost (?? ???????):
* [?????? ??? CatBoost](https://youtu.be/9ZrfErvm97M)
* [?????? ???????? ?? ??????? CatBoost (???????????? ?????)](https://youtu.be/wQt4kgAOgV0) 

## Fit

In [ ]:
model = CatBoostRegressor(iterations = 5000,
                          random_seed = RANDOM_SEED,
                          eval_metric='MAPE',
                          custom_metric=['R2', 'MAE'],
                          silent=True,
                         )
model.fit(X_train, y_train,
         #cat_features=cat_features_ids,
         eval_set=(X_test, y_test),
         verbose_eval=0,
         use_best_model=True,
         #plot=True
         )

model.save_model('catboost_single_model_baseline.model')

In [ ]:
# ????????? ????????
predict = model.predict(X_test)
print(f"???????? ?????? ?? ??????? MAPE: {(mape(y_test, predict))*100:0.2f}%")

??? ??? ?????? ?? ??????, ???? ?? ?????? ???? ?????? ? ?? ???????? ????????? catboosta, ???????? ?????? ? ??????? ?????? ? 18%!

### Log Traget
????????? ????? ?????? ? ???????? - ??? ???????? ????????? ??????? ???????? ?? ???????? ?????? (?????????? ??? ????? np.log ? np.exp).    
? ????????? ?? ????? ???????????? ????? ?????????????? ?? ??????? ??????????. ???????? ??????? ?? ???? ???????, ???? ??? ?????? :) ? ???? ????? ??????, ??????? ???? ???????? ? ????? ? **parsing_unixtime**

In [ ]:
np.log(y_train)

In [ ]:
model = CatBoostRegressor(iterations = 5000,
                          random_seed = RANDOM_SEED,
                          eval_metric='MAPE',
                          custom_metric=['R2', 'MAE'],
                          silent=True,
                         )
model.fit(X_train, np.log(y_train),
         #cat_features=cat_features_ids,
         eval_set=(X_test, np.log(y_test)),
         verbose_eval=0,
         use_best_model=True,
         #plot=True
         )

model.save_model('catboost_single_model_2_baseline.model')

In [ ]:
predict_test = np.exp(model.predict(X_test))
predict_submission = np.exp(model.predict(X_sub))

In [ ]:
print(f"???????? ?????? ?? ??????? MAPE: {(mape(y_test, predict_test))*100:0.2f}%")

??? ????? ???????? ???????? ?? 15%, ? ??? ????? ?? ???

# Submission

In [ ]:
sample_submission['price'] = predict_submission
sample_submission.to_csv(f'submission_2_v{VERSION}.csv', index=False)
sample_submission.head(10)

? ????? ???????? **MAPE 27%** ?? ??!

??????? ??????? ? ?????? ????? ????????? ?? ?? ??? ???? ? ????? ????? ???????? ?? ??????? ??? ?? ??? ?????? ? ?????? ????? ??? ???????? ? ?? ????? ?????????.

# What's next?
??? ??? ??? ????? ???????, ???? ???????? ?????????:

* ???????? ?????? ?????? 
* ??????????, ??? ????? ??????? ?? ????????? ??? ??? ??? ????? ?????????? ????????
* ????????????? ????? ????????
* ??????????? ????????? ????????? ??????
* ??????????? ?????? ????????? ? ?????????? ML
* ??????? ???????? ???????, Blending, Stacking

????????? ??? ????: https://docs.google.com/spreadsheets/d/1I_ErM3U0Cs7Rs1obyZbIEGtVn-H47pHNCi4xdDgUmXY/edit?usp=sharing